---
title : "Talk with WISKI"
execute:
    error: false
    eval: false
    output: asis
format:
    html:
        warning: false
        error: false
        code-fold: false
        css: style.css
        panel: center
---

# Quality Code

In [1]:
# zorg dat alle variables goed staan
from setup_quarto import setup_dot_env

ctx = setup_dot_env()

In [2]:
from talk_with_wiski.utils.logging_utils import setup_logging, logger
from pathlib import Path

logging_path = setup_logging(log_level="WARN", log_dir=str(Path.cwd()))
# bijvoobeeld @Andries:
logger.info(f"Logging path: {logging_path}")

In [3]:
# to do analysis on timeseries
from talk_with_wiski.models import (
    AnalyseTimeseriesQualityCodeInput,
)
from talk_with_wiski.tools import (
    get_timeseries_list,
    analyse_timeseries_quality_code,
)

import pandas as pd

In [4]:
# for statistics and trend, same object can be used
analyse_timeseries_input = AnalyseTimeseriesQualityCodeInput(
    ts_id=["37481042", "63160042"],  # belongs to stat_no "1198_WTH_BOV"
    from_date="2023-11-01",
    period="P1D",  # dit mag niet icm from_date.. to_date
    max_number_of_tsids_to_retrieve=20,
    max_number_of_values_per_tsid=1000000,
)
out_qualcode = await analyse_timeseries_quality_code(ctx, analyse_timeseries_input)
df = pd.DataFrame(out_qualcode.timeseries_quality_codes)
print(df)

      ts_id  station_name    station_no parametertype_name  n_datapoints  \
0  37481042  Bovenstrooms  1198_WTH_BOV            WNS8487            25   
1  63160042         Pomp1  BRE0301_PMP1            WNS9030            22   

   Ongecontroleerd  AutoValide (oud)  
0               25                 0  
1                0                22  


# Metadata voor TimeseriesType

In [5]:
ts_type_id_in = [6042, 14042]  # zijn de type_ids voor WNS*

In [6]:
# zorg dat alle variable goed staan
from setup_quarto import setup_dot_env

ctx = setup_dot_env()

In [7]:
from talk_with_wiski.models import GetTimeseriesTypeInput, GetTimeseriesTypeOutput
from talk_with_wiski.tools import get_timeseries_type

for idx in [0, 1]:
    get_timeseries_type_input = GetTimeseriesTypeInput(ts_type_id=ts_type_id_in[idx])
    get_timeseries_type_output: GetTimeseriesTypeOutput = await get_timeseries_type(
        ctx, get_timeseries_type_input
    )
    data = get_timeseries_type_output.model_dump()

    unique_ts_names = {x["ts_name"] for x in data["timeseries_type_list"]}
    print(unique_ts_names)

{'ContinueMeting.F', 'ContinueMeting.P', 'Handmeting', 'Handmeting.B4', 'Handmeting.B2'}
{'Jaar.Gem', 'Day.Mean', 'Uur.Gem', 'Dag.Gem', 'Maand.Gem'}


# Metadata voor {stations, sites, ts_id}
# obv request naar get*List, met returnfields die metadata bevatten
# mss moet ts_type_id hier ook nog bij? 

In [8]:
from talk_with_wiski.models import GetTimeseriesListInput, GetTimeseriesListOutput

get_timeseries_list_input = GetTimeseriesListInput(
    # parametertype_name='WNS*',  # dit geeft een leeg resultaat
    # parametertype_name=['WNS9688','WNS1919','WNS8487'], # en ook voor een lijst lijkt dit niet te werken
    parametertype_name=["WNS8487"],
    returnfields="station_name,station_no,station_id,ts_id,ts_name,parametertype_id,parametertype_name,ts_type_id,ts_type_name",
)
get_timeseries_list_output: GetTimeseriesListOutput = await get_timeseries_list(
    ctx, get_timeseries_list_input
)

data = get_timeseries_list_output.model_dump()

ts_id_list = list({x["ts_id"] for x in data["data"]})
print(ts_id_list)
ts_id_list = ts_id_list[:10]  # reduce to first 10

ts_ids_string = ",".join(ts_id_list)
print(ts_ids_string)

['31674042', '1777042', '1935042', '8574042', '2738042', '38681042', '2103042', '2321042', '8868042', '32869042', '892042', '9419042', '17101042', '2229042', '37315042', '79686042', '9429042', '37785042', '59901042', '38078042', '62136042', '86861042', '2266042', '59279042', '74801042', '80442042', '5313042', '706042', '3664042', '9427042', '38021042', '4990042', '699042', '3887042', '37951042', '81153042', '572042', '2148042', '3558042', '8505042', '19368042', '37891042', '3321042', '25339042', '38481042', '7937042', '59991042', '3310042', '1278042', '38275042', '8225042', '9399042', '60435042', '60177042', '62133042', '8662042', '37917042', '54939042', '73849042', '734042', '3358042', '60861042', '7986042', '4493042', '92931042', '81999042', '3451042', '59384042', '1732042', '75594042', '8786042', '37829042', '61383042', '2192042', '93815042', '209042', '94359042', '9327042', '60906042', '84042', '86038042', '449042', '81334042', '8513042', '80328042', '9345042', '60223042', '8099204

In [9]:
from talk_with_wiski.models import GetMetadataInput, GetMetadataOutput
from talk_with_wiski.tools import get_metadata

input = GetMetadataInput(
    ts_id=ts_id_list,
)
ts_metadata_list_of_dicts: GetMetadataOutput = await get_metadata(ctx, input)

print(ts_metadata_list_of_dicts.data[0])

# Extract site_no values from the metadata
site_no_list = [item["site_no"] for item in ts_metadata_list_of_dicts.data]
print(f"sites: {site_no_list}")

# Extract site_no values from the metadata
station_no_list = [item["station_no"] for item in ts_metadata_list_of_dicts.data]
print(f"stations: {station_no_list}")

input = GetMetadataInput(
    site_no=site_no_list,
)
site_metadata_list_of_dicts: GetMetadataOutput = await get_metadata(ctx, input)
print(site_metadata_list_of_dicts.data)

input = GetMetadataInput(
    station_no=station_no_list,
)
station_metadata_list_of_dicts: GetMetadataOutput = await get_metadata(ctx, input)
print(station_metadata_list_of_dicts.data)

# Voorbeeld voor koppelen van twee tabellen metadata (site_no based)
merged_list = [
    {**d1, **d2}
    for d1 in ts_metadata_list_of_dicts.data
    for d2 in site_metadata_list_of_dicts.data
    if d1["site_no"] == d2["site_no"]
]
print(merged_list[0])

print(len(ts_id_list), len(site_no_list), len(merged_list))

{'ts_id': '1935042', 'site_no': '4005', 'station_no': '4005_WTH_IN', 'catchment_id': '0', 'Meetdoel_BOS_hoog_water': 'no', 'STATION_NUMMER_OUD': '4005_WTH_IN', 'station_diary': '', 'Meetdoel_Mark-Vliet_boezem': 'no', 'station_gauge_datum': '', 'station_gauge_datum_unit': '', 'Meetdoel_basismeetnet_freatisch_G': 'no', 'Meetdoel_project_onderzoek': 'no', 'Meetdoel_KRW': 'no', 'Meetdoel_peilbeheer': 'yes', 'STATUS': 'Observed', 'ngr_letter': '-', 'Meetdoel_calamiteitenzorg': 'no', 'admin_level': 'District', 'region_district': '---', 'Meetdoel_Waterveiligheid_projecten': 'no', 'Meetdoel_Waterveiligheid_primair': 'no', 'Meetdoel_controlemetingen': 'no', 'Meetdoel_BMV': 'no', 'station_gauge_datum_from': '', 'Meetdoel_ondergrens_N2000': 'no', 'Meetdoel_Bedienen': 'no', 'Meetdoel_basismeetnet_N': 'no', 'MEETWIJZEBRON': 'Telemetrie TMX', 'station_diary_status': '', 'GAUGE_DATUM': '', 'Meetdoel_extern': 'no', 'Meetdoel_waterakkoorden': 'no', 'Meetdoel_Waterkwaliteit': 'no', 'Meetdoel_Waterveilig

In [27]:
# all_keys = set().union(*(d.keys() for d in ts_metadata_list_of_dicts.data))
# all_keys = set().union(*(d.keys() for d in site_metadata_list_of_dicts.data))
all_keys = set().union(*(d.keys() for d in station_metadata_list_of_dicts.data))
all_keys

{'GAUGE_DATUM',
 'GWREF_DATUM',
 'MEETWIJZEBRON',
 'Meetdoel_BMV',
 'Meetdoel_BOS_hoog_water',
 'Meetdoel_Bedienen',
 'Meetdoel_KRW',
 'Meetdoel_Mark-Vliet_boezem',
 'Meetdoel_Waterketen',
 'Meetdoel_Waterkwaliteit',
 'Meetdoel_Waterveiligheid_primair',
 'Meetdoel_Waterveiligheid_projecten',
 'Meetdoel_Waterveiligheid_regionaal',
 'Meetdoel_basismeetnet_N',
 'Meetdoel_basismeetnet_Q',
 'Meetdoel_basismeetnet_freatisch_G',
 'Meetdoel_calamiteitenzorg',
 'Meetdoel_controlemetingen',
 'Meetdoel_extern',
 'Meetdoel_ondergrens_N2000',
 'Meetdoel_peilbeheer',
 'Meetdoel_project_onderzoek',
 'Meetdoel_waterakkoorden',
 'Meetfrequentie_Handmeting',
 'PeilvakNummer',
 'STATION_NUMMER_OUD',
 'STATUS',
 'admin_level',
 'admin_name',
 'area_id',
 'area_name',
 'ngr_letter',
 'object_type',
 'object_type_shortname',
 'region_district',
 'river_id',
 'river_name',
 'station_diary',
 'station_diary_status',
 'station_gauge_datum',
 'station_gauge_datum_from',
 'station_gauge_datum_unit',
 'station_no

In [37]:
mydict = ts_metadata_list_of_dicts.data[0]
mydict
for key in sorted(mydict):
    value = mydict[key]
    if isinstance(value, str):
        value = f"'{value}'"
    print(f"'{key}': {value}")

'GAUGE_DATUM': ''
'GWREF_DATUM': ''
'MEETWIJZEBRON': 'Telemetrie TMX'
'Meetdoel_BMV': 'no'
'Meetdoel_BOS_hoog_water': 'no'
'Meetdoel_Bedienen': 'no'
'Meetdoel_KRW': 'no'
'Meetdoel_Mark-Vliet_boezem': 'no'
'Meetdoel_Waterketen': 'no'
'Meetdoel_Waterkwaliteit': 'no'
'Meetdoel_Waterveiligheid_primair': 'no'
'Meetdoel_Waterveiligheid_projecten': 'no'
'Meetdoel_Waterveiligheid_regionaal': 'no'
'Meetdoel_basismeetnet_N': 'no'
'Meetdoel_basismeetnet_Q': 'no'
'Meetdoel_basismeetnet_freatisch_G': 'no'
'Meetdoel_calamiteitenzorg': 'no'
'Meetdoel_controlemetingen': 'no'
'Meetdoel_extern': 'no'
'Meetdoel_ondergrens_N2000': 'no'
'Meetdoel_peilbeheer': 'yes'
'Meetdoel_project_onderzoek': 'no'
'Meetdoel_waterakkoorden': 'no'
'Meetfrequentie_Handmeting': '3 maanden'
'PeilvakNummer': ''
'STATION_NUMMER_OUD': '4005_WTH_IN'
'STATUS': 'Observed'
'admin_level': 'District'
'admin_name': '---'
'catchment_id': '0'
'ngr_letter': '-'
'region_district': '---'
'site_no': '4005'
'station_diary': ''
'station_diary_

In [10]:
from talk_with_wiski.models import GetTimeseriesListInput, GetTimeseriesListOutput

get_timeseries_list_input = GetTimeseriesListInput(
    parametertype_name="WNS*",
    # parametertype_name=['WNS9688','WNS1919','WNS8487'],
    # parametertype_name=["WNS8487"],
    returnfields="station_name,station_no,station_id,ts_id,ts_name,parametertype_id,parametertype_name,ts_type_id,ts_type_name",
)
get_timeseries_list_output: GetTimeseriesListOutput = await get_timeseries_list(
    ctx, get_timeseries_list_input
)

data = get_timeseries_list_output.model_dump()

ts_id_list = list({x["ts_id"] for x in data["data"]})
print(ts_id_list)
ts_id_list = ts_id_list[:10]  # reduce to first 10

ts_ids_string = ",".join(ts_id_list)
print(ts_ids_string)

['41211042', '17292042', '88456042', '1935042', '35849042', '2738042', '4744042', '92437042', '91074042', '89200042', '93090042', '9419042', '6411042', '88924042', '37315042', '5309042', '49581042', '62136042', '39654042', '85845042', '44300042', '26978042', '71936042', '59279042', '5313042', '706042', '47867042', '9427042', '23406042', '86410042', '75341042', '42117042', '23792042', '3558042', '21485042', '7937042', '6623042', '35431042', '86002042', '93274042', '26439042', '71362042', '42107042', '71474042', '65448042', '26691042', '47214042', '26510042', '88475042', '33627042', '93722042', '23384042', '69281042', '65359042', '86520042', '92931042', '81999042', '59341042', '77653042', '29236042', '72239042', '35838042', '50034042', '86142042', '39461042', '40385042', '8786042', '69732042', '209042', '38003042', '94359042', '28293042', '24448042', '93276042', '51485042', '8513042', '37256042', '80328042', '28098042', '66175042', '1536042', '44783042', '40583042', '49196042', '44853042